In [ ]:
import pandas as pd
# train test split
from sklearn.model_selection import train_test_split

df = pd.read_csv("/home/gkulemeyer/Documents/DiskB/Guillermo/revisitingCV/data/bpRNA/bpRNA_ACGU.csv")

In [ ]:
is_bprna = df["id"].str.startswith("bpRNA_")

df_bprna = df[is_bprna].copy() 
df_new = df[~is_bprna].copy() 

In [ ]:
save_cols = ["fold_name","fold_number","partition","id"]
df = pd.DataFrame(columns=save_cols) 
df_bprna["partition"] = "train"
df_bprna["fold_name"] = "bprna"
df_bprna["fold_number"] = 0


df_new["partition"] = "test"
df_new["fold_name"] = "bprna-new"
df_new["fold_number"] = 0
df = pd.concat([df, df_bprna[save_cols], df_new[save_cols]], axis=0)


In [ ]:
df.to_csv("splits/fam_folds_train_split.csv", index=False)
# 
df.groupby(["fold_name"]).count()

In [ ]:
rfam = pd.read_csv("../databases/rfam_14_2/bprna-new_ID_to_Type_Rfam14_2.csv")
df_newf = rfam.query('id in @df_new.id')
 
folds = []

top_fams = df_newf["fam"].value_counts().head(20).index
train_ids = df_bprna["id"].to_numpy()

for fold_number, fam in enumerate(top_fams):
    test_ids = df_newf.loc[df_newf["fam"] == fam, "id"].to_numpy()

    folds.append(pd.DataFrame({
        "fold_name": fam,
        "fold_number": fold_number,
        "partition": "test",
        "id": test_ids,
    }))
    # TO ADD TRAIN IN EACH SPLIT
    
    # folds.append(pd.DataFrame({
    #     "fold_name": fam,
    #     "fold_number": fold_number,
    #     "partition": "train",
    #     "id": train_ids,
    # }))

df_fam_split = pd.concat(folds, ignore_index=True)
df_fam_split.to_csv("splits/fam_folds_test_split.csv", index=False)

In [ ]:
df_fam_split